# Visualize Preprocessed DNS Snapshots (Static Plots)

This notebook loads a subsampled `.npz` file and provides tools to visualize the data across different spatial and temporal planes. 

**Key Features:**
- **Consistent Color Scales:** For each plot type (e.g., X-Z slices), the color palette is synchronized across all velocity components (u, v, w) for accurate visual comparison.
- **Grouped Parameters:** You only need to set the slice/line parameters once for each group of plots.
- **Individual Execution:** You can still run a single cell to regenerate a plot for just one velocity component after changing the shared parameters.

### 1. Setup and Imports

In [ ]:
import numpy as np
from pathlib import Path
import logging

# Import all necessary plotting functions from your core module
from mhd_surrogate_core.plot import (
    plot_xz_slice, 
    plot_xy_slice, 
    plot_yz_slice,
    plot_z_time_evolution,
    plot_x_time_evolution,
    plot_y_time_evolution
)

### 2. User Configuration

**Action Required:** Set your parameters in this cell. You can re-run this cell anytime to apply changes without reloading the data.

In [ ]:
# <<< 1. SET DATA FILE PATH >>>
# This is only used when the 'Load Data' cell is run.
data_file_path = Path("/raid/skowronek/preprocessed_dns_output/01-Cold_Runs/01-Re16K_Ha325/T1220_x1151_y5_z127_c3/T1220_x1151_y5_z127_c3.npz")

# <<< 2. DEFINE CHANNEL NAME MAPPING >>>
# Maps data channel names to display names for plots.
channel_map = {
    'vx': 'u',
    'vy': 'v',
    'vz': 'w'
    # You can add other mappings here, e.g., 'bx': 'Bx'
}

# <<< 3. SET GLOBAL PLOT SIZING PARAMETERS >>>
# Controls the overall size of the longest dimension of the dynamic plots.
global_base_size = 20
# Controls the minimum size of the shortest dimension of the dynamic plots.
global_min_size = 0.1

# <<< 4. SET UNIT LABEL FOR COLORBAR >>>
# E.g., "m/s", "T", etc. Set to None or "" for no unit.
global_unit_label = None

print("Configuration set. You can now run the 'Load Data' cell if you haven't already.")

### 3. Load Data

**Run this cell only once** at the beginning of your session or after changing the `data_file_path` above.

In [ ]:
# --- Data Loading and Initialization ---
timeseries_data = None
coords = {}

if 'data_file_path' not in locals() or not data_file_path.exists():
    logging.error(f"ERROR: Data file not found at {globals().get('data_file_path', 'Not Set')}. Please set the correct path in the configuration cell.")
else:
    with np.load(data_file_path, allow_pickle=True) as data:
        timeseries_data = data['timeseries']
        coords = {
            'labels': list(data['labels']),
            'x': data['x_coords'],
            'y': data['y_coords'],
            'z': data['z_coords'],
        }
        # Assumes data shape is (time, x, y, z, channel)
        max_time_index = timeseries_data.shape[0] - 1
        max_x_index = timeseries_data.shape[1] - 1
        max_y_index = timeseries_data.shape[2] - 1
        max_z_index = timeseries_data.shape[3] - 1
    
    print("Data loaded successfully into memory.")
    print(f"Available channels: {coords['labels']}")
    print(f"Max indices -> Time: {max_time_index}, X: {max_x_index}, Y: {max_y_index}, Z: {max_z_index}")

# Automatically identify velocity components to be plotted based on the channel_map
velocity_components = sorted([c for c in coords.get('labels', []) if c in channel_map])
if not velocity_components:
    logging.warning("Warning: No velocity components matching the channel_map found in the data.")
else:
    print(f"Identified velocity components to plot: {velocity_components}")

---

### 4. Spatial Slice Plots (2D)

#### 4.1 X-Z Slices (at constant Y)

In [ ]:
# === Parameters and Color Scale for X-Z Slices ===
# <<< MODIFY THESE VALUES >>>
plot_y_index_xz = max_y_index // 2
plot_time_index_xz = max_time_index // 2
# ---

vmin_xz, vmax_xz = None, None
if timeseries_data is not None and velocity_components:
    # Get slices for all velocity components
    slices = []
    for vc in velocity_components:
        vc_idx = coords['labels'].index(vc)
        slices.append(timeseries_data[plot_time_index_xz, :, plot_y_index_xz, :, vc_idx])
    
    # Calculate global min and max across these slices
    vmin_xz = min(s.min() for s in slices)
    vmax_xz = max(s.max() for s in slices)
    print(f"Global color scale for X-Z slices (y={plot_y_index_xz}, t={plot_time_index_xz}): [{vmin_xz:.3f}, {vmax_xz:.3f}]")

In [ ]:
# === Plot X-Z Slice for u (vx) ===
if 'vx' in velocity_components:
    plot_xz_slice(
        data_path=None, # Data is in memory
        timeseries_data=timeseries_data,
        coords=coords,
        channel='vx',
        y_index=plot_y_index_xz,
        time_index=plot_time_index_xz,
        channel_alias=channel_map.get('vx'),
        vmin=vmin_xz,
        vmax=vmax_xz,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

In [ ]:
# === Plot X-Z Slice for v (vy) ===
if 'vy' in velocity_components:
    plot_xz_slice(
        data_path=None,
        timeseries_data=timeseries_data,
        coords=coords,
        channel='vy',
        y_index=plot_y_index_xz,
        time_index=plot_time_index_xz,
        channel_alias=channel_map.get('vy'),
        vmin=vmin_xz,
        vmax=vmax_xz,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

In [ ]:
# === Plot X-Z Slice for w (vz) ===
if 'vz' in velocity_components:
    plot_xz_slice(
        data_path=None,
        timeseries_data=timeseries_data,
        coords=coords,
        channel='vz',
        y_index=plot_y_index_xz,
        time_index=plot_time_index_xz,
        channel_alias=channel_map.get('vz'),
        vmin=vmin_xz,
        vmax=vmax_xz,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

#### 4.2 X-Y Slices (at constant Z)

In [ ]:
# === Parameters and Color Scale for X-Y Slices ===
# <<< MODIFY THESE VALUES >>>
plot_z_index_xy = max_z_index // 2
plot_time_index_xy = max_time_index // 2
# ---

vmin_xy, vmax_xy = None, None
if timeseries_data is not None and velocity_components:
    slices = []
    for vc in velocity_components:
        vc_idx = coords['labels'].index(vc)
        slices.append(timeseries_data[plot_time_index_xy, :, :, plot_z_index_xy, vc_idx])
    
    vmin_xy = min(s.min() for s in slices)
    vmax_xy = max(s.max() for s in slices)
    print(f"Global color scale for X-Y slices (z={plot_z_index_xy}, t={plot_time_index_xy}): [{vmin_xy:.3f}, {vmax_xy:.3f}]")

In [ ]:
# === Plot X-Y Slice for u (vx) ===
if 'vx' in velocity_components:
    plot_xy_slice(
        data_path=None,
        timeseries_data=timeseries_data,
        coords=coords,
        channel='vx',
        z_index=plot_z_index_xy,
        time_index=plot_time_index_xy,
        channel_alias=channel_map.get('vx'),
        vmin=vmin_xy,
        vmax=vmax_xy,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

In [ ]:
# === Plot X-Y Slice for v (vy) ===
if 'vy' in velocity_components:
    plot_xy_slice(
        data_path=None,
        timeseries_data=timeseries_data,
        coords=coords,
        channel='vy',
        z_index=plot_z_index_xy,
        time_index=plot_time_index_xy,
        channel_alias=channel_map.get('vy'),
        vmin=vmin_xy,
        vmax=vmax_xy,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

In [ ]:
# === Plot X-Y Slice for w (vz) ===
if 'vz' in velocity_components:
    plot_xy_slice(
        data_path=None,
        timeseries_data=timeseries_data,
        coords=coords,
        channel='vz',
        z_index=plot_z_index_xy,
        time_index=plot_time_index_xy,
        channel_alias=channel_map.get('vz'),
        vmin=vmin_xy,
        vmax=vmax_xy,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

#### 4.3 Y-Z Slices (at constant X)

In [ ]:
# === Parameters and Color Scale for Y-Z Slices ===
# <<< MODIFY THESE VALUES >>>
plot_x_index_yz = max_x_index // 2
plot_time_index_yz = max_time_index // 2
# ---

vmin_yz, vmax_yz = None, None
if timeseries_data is not None and velocity_components:
    slices = []
    for vc in velocity_components:
        vc_idx = coords['labels'].index(vc)
        slices.append(timeseries_data[plot_time_index_yz, plot_x_index_yz, :, :, vc_idx])
    
    vmin_yz = min(s.min() for s in slices)
    vmax_yz = max(s.max() for s in slices)
    print(f"Global color scale for Y-Z slices (x={plot_x_index_yz}, t={plot_time_index_yz}): [{vmin_yz:.3f}, {vmax_yz:.3f}]")

In [ ]:
# === Plot Y-Z Slice for u (vx) ===
if 'vx' in velocity_components:
    plot_yz_slice(
        data_path=None,
        timeseries_data=timeseries_data,
        coords=coords,
        channel='vx',
        x_index=plot_x_index_yz,
        time_index=plot_time_index_yz,
        channel_alias=channel_map.get('vx'),
        vmin=vmin_yz,
        vmax=vmax_yz,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

In [ ]:
# === Plot Y-Z Slice for v (vy) ===
if 'vy' in velocity_components:
    plot_yz_slice(
        data_path=None,
        timeseries_data=timeseries_data,
        coords=coords,
        channel='vy',
        x_index=plot_x_index_yz,
        time_index=plot_time_index_yz,
        channel_alias=channel_map.get('vy'),
        vmin=vmin_yz,
        vmax=vmax_yz,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

In [ ]:
# === Plot Y-Z Slice for w (vz) ===
if 'vz' in velocity_components:
    plot_yz_slice(
        data_path=None,
        timeseries_data=timeseries_data,
        coords=coords,
        channel='vz',
        x_index=plot_x_index_yz,
        time_index=plot_time_index_yz,
        channel_alias=channel_map.get('vz'),
        vmin=vmin_yz,
        vmax=vmax_yz,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

---

### 5. Time Evolution Plots (1D Line over Time)

#### 5.1 Evolution along Z-axis (at constant X, Y)

In [ ]:
# === Parameters and Color Scale for Time-Z Plots ===
# <<< MODIFY THESE VALUES >>>
plot_x_index_tz = max_x_index // 2
plot_y_index_tz = max_y_index // 2
# ---

vmin_tz, vmax_tz = None, None
if timeseries_data is not None and velocity_components:
    slices = []
    for vc in velocity_components:
        vc_idx = coords['labels'].index(vc)
        slices.append(timeseries_data[:, plot_x_index_tz, plot_y_index_tz, :, vc_idx])
    
    vmin_tz = min(s.min() for s in slices)
    vmax_tz = max(s.max() for s in slices)
    print(f"Global color scale for Time-Z plots (x={plot_x_index_tz}, y={plot_y_index_tz}): [{vmin_tz:.3f}, {vmax_tz:.3f}]")

In [ ]:
# === Plot Time-Z Evolution for u (vx) ===
if 'vx' in velocity_components:
    plot_z_time_evolution(
        data_path=None,
        timeseries_data=timeseries_data,
        coords=coords,
        channel='vx',
        x_index=plot_x_index_tz,
        y_index=plot_y_index_tz,
        channel_alias=channel_map.get('vx'),
        vmin=vmin_tz,
        vmax=vmax_tz,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

In [ ]:
# === Plot Time-Z Evolution for v (vy) ===
if 'vy' in velocity_components:
    plot_z_time_evolution(
        data_path=None,
        timeseries_data=timeseries_data,
        coords=coords,
        channel='vy',
        x_index=plot_x_index_tz,
        y_index=plot_y_index_tz,
        channel_alias=channel_map.get('vy'),
        vmin=vmin_tz,
        vmax=vmax_tz,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

In [ ]:
# === Plot Time-Z Evolution for w (vz) ===
if 'vz' in velocity_components:
    plot_z_time_evolution(
        data_path=None,
        timeseries_data=timeseries_data,
        coords=coords,
        channel='vz',
        x_index=plot_x_index_tz,
        y_index=plot_y_index_tz,
        channel_alias=channel_map.get('vz'),
        vmin=vmin_tz,
        vmax=vmax_tz,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

#### 5.2 Evolution along X-axis (at constant Y, Z)

In [ ]:
# === Parameters and Color Scale for Time-X Plots ===
# <<< MODIFY THESE VALUES >>>
plot_y_index_tx = max_y_index // 2
plot_z_index_tx = max_z_index // 2
# ---

vmin_tx, vmax_tx = None, None
if timeseries_data is not None and velocity_components:
    slices = []
    for vc in velocity_components:
        vc_idx = coords['labels'].index(vc)
        slices.append(timeseries_data[:, :, plot_y_index_tx, plot_z_index_tx, vc_idx])
    
    vmin_tx = min(s.min() for s in slices)
    vmax_tx = max(s.max() for s in slices)
    print(f"Global color scale for Time-X plots (y={plot_y_index_tx}, z={plot_z_index_tx}): [{vmin_tx:.3f}, {vmax_tx:.3f}]")

In [ ]:
# === Plot Time-X Evolution for u (vx) ===
if 'vx' in velocity_components:
    plot_x_time_evolution(
        data_path=None,
        timeseries_data=timeseries_data,
        coords=coords,
        channel='vx',
        y_index=plot_y_index_tx,
        z_index=plot_z_index_tx,
        channel_alias=channel_map.get('vx'),
        vmin=vmin_tx,
        vmax=vmax_tx,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

In [ ]:
# === Plot Time-X Evolution for v (vy) ===
if 'vy' in velocity_components:
    plot_x_time_evolution(
        data_path=None,
        timeseries_data=timeseries_data,
        coords=coords,
        channel='vy',
        y_index=plot_y_index_tx,
        z_index=plot_z_index_tx,
        channel_alias=channel_map.get('vy'),
        vmin=vmin_tx,
        vmax=vmax_tx,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

In [ ]:
# === Plot Time-X Evolution for w (vz) ===
if 'vz' in velocity_components:
    plot_x_time_evolution(
        data_path=None,
        timeseries_data=timeseries_data,
        coords=coords,
        channel='vz',
        y_index=plot_y_index_tx,
        z_index=plot_z_index_tx,
        channel_alias=channel_map.get('vz'),
        vmin=vmin_tx,
        vmax=vmax_tx,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

#### 5.3 Evolution along Y-axis (at constant X, Z)

In [ ]:
# === Parameters and Color Scale for Time-Y Plots ===
# <<< MODIFY THESE VALUES >>>
plot_x_index_ty = max_x_index // 2
plot_z_index_ty = max_z_index // 2
# ---

vmin_ty, vmax_ty = None, None
if timeseries_data is not None and velocity_components:
    slices = []
    for vc in velocity_components:
        vc_idx = coords['labels'].index(vc)
        slices.append(timeseries_data[:, plot_x_index_ty, :, plot_z_index_ty, vc_idx])
    
    vmin_ty = min(s.min() for s in slices)
    vmax_ty = max(s.max() for s in slices)
    print(f"Global color scale for Time-Y plots (x={plot_x_index_ty}, z={plot_z_index_ty}): [{vmin_ty:.3f}, {vmax_ty:.3f}]")

In [ ]:
# === Plot Time-Y Evolution for u (vx) ===
if 'vx' in velocity_components:
    plot_y_time_evolution(
        data_path=None,
        timeseries_data=timeseries_data,
        coords=coords,
        channel='vx',
        x_index=plot_x_index_ty,
        z_index=plot_z_index_ty,
        channel_alias=channel_map.get('vx'),
        vmin=vmin_ty,
        vmax=vmax_ty,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

In [ ]:
# === Plot Time-Y Evolution for v (vy) ===
if 'vy' in velocity_components:
    plot_y_time_evolution(
        data_path=None,
        timeseries_data=timeseries_data,
        coords=coords,
        channel='vy',
        x_index=plot_x_index_ty,
        z_index=plot_z_index_ty,
        channel_alias=channel_map.get('vy'),
        vmin=vmin_ty,
        vmax=vmax_ty,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )

In [ ]:
# === Plot Time-Y Evolution for w (vz) ===
if 'vz' in velocity_components:
    plot_y_time_evolution(
        data_path=None,
        timeseries_data=timeseries_data,
        coords=coords,
        channel='vz',
        x_index=plot_x_index_ty,
        z_index=plot_z_index_ty,
        channel_alias=channel_map.get('vz'),
        vmin=vmin_ty,
        vmax=vmax_ty,
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label
    )